# 7 · Treino de segmentação das faixas

Treina uma UNet++/EfficientNet-B2 sobre máscaras HiRes do subconjunto Barrettotte. Os grupos de valor/tolerância seguem a divisão do notebook 2: um grupo inteiro permanece em uma única partição. Arquivos que não podem ser vinculados de forma inequívoca ao Barrettotte são excluídos; isso evita incluir por engano imagens Eralp no treino deste experimento.

A inicialização usa apenas pesos genéricos ImageNet, não o segmentador HiRes antigo. A identidade física das peças ainda não está documentada; a separação por grupo de valor não é prova de independência física. O teste de leitura abaixo começa com recortes já fornecidos e não mede detecção de objetos.

In [ ]:
from pathlib import Path
import os,sys,json,hashlib,re,random,shutil
from collections import Counter
import cv2,numpy as np,pandas as pd,torch
from PIL import Image
from torch.utils.data import DataLoader
import segmentation_models_pytorch as smp

PROJECT=Path(os.environ.get("RESISTOR_PROJECT_ROOT","..")).expanduser().resolve()
DATA=Path(os.environ.get("RESISTOR_DATA_ROOT",str(PROJECT/"data"))).expanduser().resolve()
OUTPUT=Path(os.environ.get("RESISTOR_OUTPUT",str(PROJECT/"outputs/proposta"))).expanduser().resolve()
HIRES=DATA/"sources/github-hires"
sys.path.insert(0,str(HIRES/"segmentation"));sys.path.insert(0,str(HIRES/"band_extraction"))
from dataset import ResistorSegDataset,get_train_transform,get_val_transform,NUM_CLASSES
from infer import _run_pass
from resistance_calculator import calculate_resistance_with_axis_info
from color_code_tables import ResistanceResult
DEVICE=torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
SEED=42;random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
torch.set_num_threads(4)
print("Dispositivo:",DEVICE)


## Preparação e separação por grupos

Somente nomes da forma `4B-...-T...-N_jpg.rf....` ou `5B-...-T...-N_jpg.rf....` são aceitos. Os grupos e sua divisão são derivados do manifesto Barrettotte; imagens e máscaras do HiRes são copiadas ou ligadas apenas dentro da pasta de resultados.

In [ ]:
manifest=PROJECT/"data/manifests/barrettotte-crop-value.jsonl"
rows=[json.loads(s) for s in manifest.read_text(encoding="utf-8").splitlines() if s.strip()]
groups=sorted({Path(r["path"]).parent.name for r in rows},key=lambda g:hashlib.sha256(("42:"+g).encode()).hexdigest())
part={g:("train" if i<int(.70*len(groups)) else "val" if i<int(.85*len(groups)) else "test") for i,g in enumerate(groups)}
expected={}
for row in rows:
    group=Path(row["path"]).parent.name
    value=row["objects"][0]
    if group in expected and expected[group]["ohms"]!=value["ohms"]:raise ValueError(f"Valor conflitante: {group}")
    expected[group]=value
source=HIRES/"datasets/segmentation"; prepared=OUTPUT/"faixas-agrupadas"
pattern=re.compile(r"^([45]B-.+-T[0-9]+)-[0-9]+_jpg\.rf\.",re.I)
index=[]
for split in ("train","val"):
    for image in sorted((source/split/"images").iterdir()):
        match=pattern.match(image.name)
        if not match or match.group(1) not in part:continue
        group=match.group(1);mask=source/split/"masks"/(image.stem+".png")
        if not mask.exists():continue
        dest=part[group]
        image_dir,mask_dir=prepared/dest/"images",prepared/dest/"masks"
        image_dir.mkdir(parents=True,exist_ok=True);mask_dir.mkdir(parents=True,exist_ok=True)
        for original,target in ((image,image_dir/image.name),(mask,mask_dir/mask.name)):
            if target.exists():
                if hashlib.sha256(original.read_bytes()).digest()!=hashlib.sha256(target.read_bytes()).digest():raise ValueError(f"Nome duplicado: {target}")
            else:
                try:os.link(original,target)
                except OSError:shutil.copy2(original,target)
        index.append({"image":str(image.relative_to(source)),"group":group,"split":dest,"ohms":expected[group]["ohms"]})
index=pd.DataFrame(index)
assert len(index)>0 and index.groupby("group")["split"].nunique().max()==1
index.to_csv(prepared/"manifesto.csv",index=False)
print(index.groupby("split").agg(imagens=("image","size"),grupos=("group","nunique")))


## Treinamento

A seleção do checkpoint usa apenas IoU médio das classes de faixa na validação. Aumento de dados é aplicado somente ao treino. É possível configurar `RESISTOR_SEG_EPOCHS`, `RESISTOR_SEG_SIZE` e `RESISTOR_SEG_BATCH`. O treino usa pesos ImageNet para o encoder e começa uma nova cabeça de segmentação.

In [ ]:
size=int(os.environ.get("RESISTOR_SEG_SIZE","256"));batch=int(os.environ.get("RESISTOR_SEG_BATCH","4"));epochs=int(os.environ.get("RESISTOR_SEG_EPOCHS","15"))
train_ds=ResistorSegDataset(str(prepared/"train/images"),str(prepared/"train/masks"),transform=get_train_transform(size),size=size)
val_ds=ResistorSegDataset(str(prepared/"val/images"),str(prepared/"val/masks"),transform=get_val_transform(),size=size)
test_ds=ResistorSegDataset(str(prepared/"test/images"),str(prepared/"test/masks"),transform=get_val_transform(),size=size)
assert min(len(train_ds),len(val_ds),len(test_ds))>0
train_loader=DataLoader(train_ds,batch_size=batch,shuffle=True,num_workers=2)
val_loader=DataLoader(val_ds,batch_size=batch,shuffle=False,num_workers=2)
test_loader=DataLoader(test_ds,batch_size=batch,shuffle=False,num_workers=2)
model=smp.UnetPlusPlus(encoder_name="efficientnet-b2",encoder_weights="imagenet",in_channels=3,classes=NUM_CLASSES,activation=None).to(DEVICE)
dice=smp.losses.DiceLoss(mode="multiclass",from_logits=True);focal=smp.losses.FocalLoss(mode="multiclass",gamma=2.)
optimizer=torch.optim.Adam(model.parameters(),lr=1e-4,weight_decay=1e-4)
existing=os.environ.get("RESISTOR_SEG_EVALUATE_WEIGHTS")
checkpoint=Path(existing).expanduser().resolve() if existing else OUTPUT/"segmentacao-faixas-best.pt"

def segmentation_metrics(loader):
    model.eval();matrix=torch.zeros((NUM_CLASSES,NUM_CLASSES),dtype=torch.int64,device=DEVICE)
    with torch.no_grad():
        for images,masks in loader:
            images,masks=images.to(DEVICE),masks.to(DEVICE)
            pred=model(images).argmax(dim=1)
            matrix+=torch.bincount((masks*NUM_CLASSES+pred).flatten(),minlength=NUM_CLASSES**2).reshape(NUM_CLASSES,NUM_CLASSES)
    m=matrix.cpu().numpy();inter=m.diagonal();union=m.sum(0)+m.sum(1)-inter
    iou=np.divide(inter,union,out=np.full(NUM_CLASSES,np.nan),where=union>0)
    return float(np.nanmean(iou[1:])),iou,m.sum(axis=1)

if existing:
    assert checkpoint.exists(),f"Checkpoint não encontrado: {checkpoint}"
    print("Reavaliando pesos fornecidos; confira a separação dos dados de treino.")
else:
    history=[];best=-1.
    for epoch in range(1,epochs+1):
        model.train();losses=[]
        for images,masks in train_loader:
            images,masks=images.to(DEVICE),masks.to(DEVICE)
            optimizer.zero_grad(set_to_none=True);logits=model(images)
            loss=.5*dice(logits,masks)+.5*focal(logits,masks)
            if not torch.isfinite(loss):raise ValueError("Perda não finita")
            loss.backward();optimizer.step();losses.append(float(loss.detach().cpu()))
        score,_,_=segmentation_metrics(val_loader)
        history.append({"epoca":epoch,"perda_treino":float(np.mean(losses)),"iou_medio_val":score})
        pd.DataFrame(history).to_csv(OUTPUT/"segmentacao_treino.csv",index=False)
        if score>best:best=score;torch.save(model.state_dict(),checkpoint)
        print(f"Época {epoch}/{epochs}: perda {np.mean(losses):.4f}, IoU val {score:.4f}")
    print("Melhor IoU de validação:",best,"checkpoint:",checkpoint)


## Teste reservado: máscaras e valores

O IoU é calculado por classe de faixa a partir de todos os pixels do teste. A leitura usa cada recorte anotado pelo conjunto HiRes: acertos e recusas são contabilizados entre todas as imagens. Isso **não** é acurácia de ponta a ponta em imagens completas.

In [ ]:
model.load_state_dict(torch.load(checkpoint,map_location=DEVICE,weights_only=True));model.eval()
miou,ious,support=segmentation_metrics(test_loader)
names=["fundo","preto","azul","marrom","dourado","verde","cinza","laranja","violeta","vermelho","prateado","branco","amarelo"]
class_scores={names[i]:None if np.isnan(ious[i]) else float(ious[i]) for i in range(1,NUM_CLASSES)}
readings=[]; transform=get_val_transform()
for image_path,_ in test_ds.pairs:
    path=Path(image_path);match=pattern.match(path.name);group=match.group(1)
    rgb=np.asarray(Image.open(path).convert("RGB"))
    mask,confidence=_run_pass(model,rgb,transform,DEVICE,size)
    result,_=calculate_resistance_with_axis_info(mask)
    value=float(result.value) if isinstance(result,ResistanceResult) else None
    tolerance=float(result.tolerance) if isinstance(result,ResistanceResult) else None
    truth=float(expected[group]["ohms"]);truth_tol=expected[group].get("tolerance_percent")
    readings.append({"imagem":path.name,"grupo":group,"valor_real":truth,"valor_previsto":value,
                     "tolerancia_real":truth_tol,"tolerancia_prevista":tolerance,
                     "correto":value is not None and np.isclose(value,truth,rtol=1e-9),
                     "tolerancia_correta":tolerance is not None and truth_tol is not None and np.isclose(tolerance,truth_tol),
                     "dentro_tolerancia":value is not None and truth_tol is not None and abs(value-truth)<=abs(truth)*truth_tol/100,
                     "segmentacao":float(confidence)})
table=pd.DataFrame(readings);table.to_csv(OUTPUT/"segmentacao_leituras_teste.csv",index=False)
emitted=table.valor_previsto.notna()
known=table.tolerancia_real.notna()
report={"imagens_teste":len(table),"iou_medio_faixas":miou,"iou_por_faixa":class_scores,
        "pixels_rotulados_por_faixa":{names[i]:int(support[i]) for i in range(1,NUM_CLASSES)},"cobertura":float(emitted.mean()),
        "valor_exato_total":float(table.correto.mean()),"valor_exato_emitidos":float(table.loc[emitted,"correto"].mean()) if emitted.any() else None,
        "tolerancia_exata_total":float(table.loc[known,"tolerancia_correta"].mean()) if known.any() else None,
        "dentro_tolerancia_total":float(table.loc[known,"dentro_tolerancia"].mean()) if known.any() else None,
        "escopo":"recortes anotados; grupos por valor; independência física não verificada"}
(OUTPUT/"segmentacao_teste.json").write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding="utf-8")
print(json.dumps(report,indent=2,ensure_ascii=False))
